In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr

# Change working directory to the notebook's directory
notebook_dir = os.path.dirname(os.path.abspath("Frozen_linear_ADAS.ipynb"))
os.chdir(notebook_dir)

# -------------------
# Config
# -------------------
n_folds = 5
train_pattern = "../../splits/adni/train_subject_list_adni_{}"  # files: train_subject_list_adni_0 ... _4
val_pattern   = "../../splits/adni/val_subject_list_adni_{}"    # files: val_subject_list_adni_0 ... _4
test_pattern  = "../../splits/adni/test_subject_list_adni_{}"   # files: test_subject_list_adni_0 ... _4


metadata_csv = "../../metadata/adni_metadata.csv"

# Feature sets (keep as list, can have multiple)
feature_sets = ["../../latents/cls_adni_k8pcq4ai_300.npz"]

# Output folder for results/plots
out_dir = "cv_results"
os.makedirs(out_dir, exist_ok=True)

# -------------------
# Load metadata (contains target)
# -------------------
df = pd.read_csv(metadata_csv)
df = df.dropna(subset=['src_subject_id','TOTAL13_z'])
age_map = dict(zip(df["src_subject_id"].astype(str), df["TOTAL13_z"].astype(float)))

def save_predictions_csv(csv_path, subject_ids, y_true, y_pred):
    df = pd.DataFrame({
        "subject_id": subject_ids,
        "y_true": y_true.astype(float),
        "y_pred": y_pred.astype(float),
    })
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)

# -------------------
# Main loop: feature sets -> folds
# -------------------
for feat_file in feature_sets:
    features_dict = np.load(feat_file)

    # Prepare list to accumulate per-fold results
    rows = []

    # Prepare plotting: one figure per feature file, 1 x n_folds subplots
    fig, axes = plt.subplots(1, n_folds, figsize=(4 * n_folds, 4), squeeze=False)
    axes = axes.ravel()

    for fold in range(n_folds):
        # Load IDs for this fold
        train_fname = train_pattern.format(fold)
        val_fname   = val_pattern.format(fold)
        test_fname  = test_pattern.format(fold)

        # NOTE: keep dtype=str so we can split
        train_ids = np.loadtxt(train_fname, dtype=str)
        val_ids   = np.loadtxt(val_fname, dtype=str)
        test_ids  = np.loadtxt(test_fname, dtype=str)

        # Collect subject IDs that are present in features and metadata
        tr_ok = [sid for sid in train_ids if sid in features_dict and sid.split('/')[0] in age_map]
        va_ok = [sid for sid in val_ids   if sid in features_dict and sid.split('/')[0] in age_map]
        test_ok = [sid for sid in test_ids if sid in features_dict and sid.split('/')[0] in age_map]

        if len(tr_ok) == 0 or len(va_ok) == 0:
            print(f"Warning: fold {fold} for {feat_file} has empty train or val after filtering. Skipping.")
            continue

        # Build feature matrices (replace NaNs with 0)
        X_train = np.vstack([features_dict[sid] for sid in tr_ok])
        X_val   = np.vstack([features_dict[sid] for sid in va_ok])
        X_test  = np.vstack([features_dict[sid] for sid in test_ok])
        X_train = np.nan_to_num(X_train, nan=0.0)
        X_val   = np.nan_to_num(X_val, nan=0.0)
        X_test  = np.nan_to_num(X_test, nan=0.0)

        # Targets aligned to the filtered IDs
        y_train = np.array([age_map[sid.split('/')[0]] for sid in tr_ok], dtype=np.float64)
        y_val   = np.array([age_map[sid.split('/')[0]] for sid in va_ok], dtype=np.float64)
        y_test  = np.array([age_map[sid.split('/')[0]] for sid in test_ok], dtype=np.float64)

        # Normalize features by training set (per-column)
        X_mean = X_train.mean(axis=0, keepdims=True)
        X_std  = X_train.std(axis=0, keepdims=True) + 1e-8
        X_train_norm = (X_train - X_mean) / X_std
        X_val_norm   = (X_val   - X_mean) / X_std
        X_test_norm  = (X_test  - X_mean) / X_std

        # Train linear regression on normalized features (targets kept in original scale)
        lr = LinearRegression()
        lr.fit(X_train_norm, y_train)

        # Predict on validation
        y_pred = lr.predict(X_val_norm)
        y_pred_norm_test = lr.predict(X_test_norm)

        # Compute metrics
        mse = mean_squared_error(y_val, y_pred)
        r2  = r2_score(y_val, y_pred)
        # pearsonr needs at least 2 non-constant samples; guard against exceptions
        try:
            rho, pval = pearsonr(y_val, y_pred)
        except Exception:
            rho, pval = np.nan, np.nan

        rows.append({
            "feature_file": os.path.basename(feat_file),
            "fold": fold,
            "n_val": len(y_val),
            "MSE": mse,
            "rho": rho,
            "pval": pval,
            "r2": r2
        })

        csv_path = f'CV_ADAS_results/frozen_linear_val_{fold}.csv'
        save_predictions_csv(csv_path, va_ok, y_val, y_pred)
        csv_path = f'CV_ADAS_results/frozen_linear_test_{fold}.csv'
        save_predictions_csv(csv_path, test_ok, y_test, y_pred_norm_test)

        # ---- Plot for this fold in the feature's figure
        ax = axes[fold]
        ax.scatter(y_val, y_pred, alpha=0.7)

        # Best-fit line (ŷ vs y) — guard polyfit
        try:
            a, b = np.polyfit(y_val, y_pred, deg=1)
            x_line = np.linspace(np.min(y_val), np.max(y_val), 100)
            y_line = a * x_line + b
            ax.plot(x_line, y_line, linestyle='--', linewidth=2)
        except Exception:
            pass

        ax.set_xlabel("Ground Truth")
        ax.set_ylabel("Predicted")
        ax.set_title(f"fold {fold}\nMSE={mse:.3f}, rho={np.nan_to_num(rho):.3f}")

        ax.text(0.02, 0.98, rf"$\rho$ = {np.nan_to_num(rho):.3f}\n$R^2$ = {r2:.3f}",
                transform=ax.transAxes, va='top', ha='left',
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="gray", alpha=0.7))

    fig.suptitle(f"Validation Predictions per Fold — {os.path.basename(feat_file)}", fontsize=12)
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Convert rows to DataFrame, compute summary stats
    df_res = pd.DataFrame(rows)
    # Save per-feature CSV
    csv_out = os.path.join(out_dir, f"cv_table_{os.path.basename(feat_file)}.csv")
    df_res.to_csv(csv_out, index=False)
    print(f"\nPer-fold results for {feat_file}:")
    print(df_res)

    # Summary statistics across folds (mean ± std) for MSE and rho
    summary = df_res.agg({"MSE": ["mean", "std"], "rho": ["mean", "std"], "r2": ["mean", "std"]})
    print("\nSummary (mean ± std) across folds:")
    print(summary)

    # Save summary
    summary_out = os.path.join(out_dir, f"cv_summary_{os.path.basename(feat_file)}.csv")
    summary.to_csv(summary_out)

    # Show the figure
    plt.show()
